# Business Entity Resolution Pipeline

Run this notebook in Google Colab. It uploads or locates the `student_resource` folder, installs dependencies, runs Stage 3 validation, generates test predictions, and validates the final submission files.

**Important:** The full dataset needs substantial RAM. Use Colab High-RAM when available. The final score cannot be guaranteed before the competition scorer evaluates `matching_results.tsv`.

In [ ]:
# Dependencies are installed after the project files are loaded below.

## 1. Load the project files

Choose one method below. Run only the method that matches where your files are.

In [ ]:
# Method A: upload a ZIP containing the student_resource folder.
# Create the ZIP locally first if needed:
# zip -r student_resource.zip student_resource
from google.colab import files
import os
import zipfile

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Upload a .zip file containing student_resource/')
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content')
if not os.path.exists('/content/student_resource/dataset/train/train_source1.tsv'):
    raise FileNotFoundError('Expected /content/student_resource/dataset after extraction')
ROOT = '/content/student_resource'
print(ROOT)

In [ ]:
# Method B: use files already stored in Google Drive instead of uploading a ZIP.
# Run this cell only if ROOT was not set by Method A.
from google.colab import drive
drive.mount('/content/drive')

# Change this path to your Drive copy.
ROOT = '/content/drive/MyDrive/student_resource'
print(ROOT)

In [ ]:
# Confirm the required files exist, then install the pinned dependencies.
from pathlib import Path

root = Path(ROOT)
required = [
    root / 'dataset/train/train_source1.tsv',
    root / 'dataset/train/train_source2.tsv',
    root / 'dataset/train/train_source3.tsv',
    root / 'dataset/train/train_ground_truth.tsv',
    root / 'dataset/test/test_source1.tsv',
    root / 'dataset/test/test_source2.tsv',
    root / 'dataset/test/test_source3.tsv',
    root / 'utils/validate_submission.py',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

!pip install -q -r {root / 'code/business_entity_resolution/requirements.txt'}
print('All required files found and dependencies installed.')

## 2. Test normalization on real rows

In [ ]:
import sys
sys.path.insert(0, str(root / 'code/business_entity_resolution'))
from src.data_io import load_sample

sample = load_sample(root / 'dataset/train/train_source1.tsv', rows=20)
display(sample[['business_name', 'business_name_norm', 'business_address', 'business_address_norm', 'city_token', 'phonetic_key']])

## 3. Run Stage 2 blocking recall evaluation

This is the first full-data run and may take a long time. It creates `output/candidate_pairs.tsv` for the training split and prints recall and candidate statistics.

In [ ]:
import os
os.chdir(root)
!PYTHONPATH=code/business_entity_resolution python -m src.pipeline stage2 --root . --split train --ground-truth dataset/train/train_ground_truth.tsv --output output

## 4. Train and select the F₀.₅ threshold

The selected threshold is learned from a group-aware validation split. Read the printed validation F₀.₅ before proceeding.

In [ ]:
!PYTHONPATH=code/business_entity_resolution python -m src.pipeline stage3 --root . --ground-truth dataset/train/train_ground_truth.tsv

## 5. Generate final test predictions

This runs blocking on the test corpus, trains on the labeled training candidates, learns the validation threshold, and writes both required output files.

In [ ]:
!PYTHONPATH=code/business_entity_resolution python -m src.pipeline stage4 --root . --ground-truth dataset/train/train_ground_truth.tsv --output output

## 6. Validate the final TSV files

A successful validation means the files satisfy the competition's formatting and coverage rules. It does not measure leaderboard accuracy.

In [ ]:
!python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test

In [ ]:
# Download the two files for submission.
from google.colab import files
files.download('output/matching_results.tsv')
files.download('output/candidate_pairs.tsv')